In this notebook we process the data from the MD-Simulations

In [5]:
import pandas as pd

# We drop all duplicate molecules and keep the best (=lowest) score
def clean_dataset(df):
    n_duplicates = df["smiles"].duplicated().sum()
    print(f"Dataset contains {n_duplicates} duplicates.")
    
    assert not df["target"].isna().any(), "Dataset contains molecules without a score."
    
    clean_df = df.sort_values(by="target").drop_duplicates(["smiles"], keep="first")
    # Avoid any information leakage by having the data ordered
    clean_df = clean_df.sample(frac=1.0, random_state=0).reset_index(drop=True)
    return clean_df

We start by cleaning the Enamine datasets that have been used in the literature to evaluate active learning.

In [6]:
for ds in ["unprocessed_Enamine10k_scores.csv", "unprocessed_Enamine50k_scores.csv"]:
    df = pd.read_csv(ds)
    clean_df = clean_dataset(df)
    
    clean_df.to_csv(ds.removeprefix("unprocessed_"), index=False)

Dataset contains 3 duplicates.
Dataset contains 7 duplicates.


The results from docking and MMGBSA/MMPBSA have been prepared in the `results.csv` file. We now extract the necessary columns for bayesian optimization.

In [7]:
results_df = pd.read_csv("results.csv")
results_df = results_df.drop(columns=["orig_smiles"])
results_df = results_df.rename(columns={"prot_smiles": "smiles"})
results_df = results_df.sample(frac=1, random_state=0)  # Shuffle to avoid any potential bias by being sorted by mmgbsa_score
results_df.head()

,name,smiles,mmgbsa_score,mmgbsa_score_sem,mmpbsa_score,mmpbsa_score_sem,vina_score
24141,ZINCtU000002cWHx,O=C(CCc1c(-c2ccccc2)[nH]c2ccccc12)Oc1ccc2cc[nH...,-32.196145,0.444212,-25.165391,0.362708,-9.789
31315,ZINCmS00000EtNsB,CCc1c(C(=O)OC2CCC(C)(C)CC2)[nH]c2ccccc12,-30.379023,0.248125,-25.584308,0.241032,-9.181
31760,ZINCnA00000l4lE9,CC[C@@H](COC)NC(=O)c1[nH]c2c(Br)cccc2c1CCCO,-30.268506,0.406560,-22.859678,0.379812,-7.441
43743,ZINCjl000003qQ1o,C[C@H](OC(=O)CCc1c[nH]c2ccccc12)C(N)=O,-26.191771,0.372821,-22.701846,0.303263,-7.966
41331,ZINCmU00000uYYK1,O=C(CCc1c[nH]c2cc(Cl)ccc12)Oc1ccccc1Br,-27.219941,0.744492,-23.106211,0.565013,-9.111


In [8]:
results_df[["name", "smiles", "mmgbsa_score"]].rename(columns={"mmgbsa_score":"target"}).to_csv("MCL1-mmgbsa.csv")
results_df[["name", "smiles", "mmpbsa_score"]].rename(columns={"mmpbsa_score":"target"}).to_csv("MCL1-mmpbsa.csv")
results_df[["name", "smiles", "vina_score"]].rename(columns={"vina_score":"target"}).to_csv("MCL1-vina.csv")